# Server CPU Temperature Prediction
Concise, code-only pipeline implementing regression modeling for server thermal telemetry.

### 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error

warnings.filterwarnings('ignore')
print('Libraries imported.')

### 2. Load Dataset & Preview

In [ ]:
df = pd.read_csv('server_cpu_dataset_nonlinear_6000_v2.csv')
print('Dataset shape:', df.shape)
df.info()
df.head()

### 3. Exploratory Data Analysis

In [ ]:
print('Summary Statistics:')
display(df.describe())
print('Missing Values:', df.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['Temperature'], bins=40, kde=True, ax=axes[0], color='crimson')
axes[0].set_title('CPU Temperature Distribution')
axes[0].set_xlabel('Temperature (C)')

sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1])
axes[1].set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

### 4. Feature Engineering & IQR Outlier Filtering

In [ ]:
# Missing value imputation
df.fillna(df.mean(numeric_only=True), inplace=True)

# Feature engineering
df['Power_Load'] = df['Voltage'] * df['Current']
df['Workload'] = df['CPU_Utilization'] * df['Memory_Usage']

features = [
    'CPU_Utilization', 'Memory_Usage', 'Clock_Speed',
    'Ambient_Temperature', 'Voltage', 'Current',
    'Power_Load', 'Workload'
]
target = 'Temperature'

# IQR outlier filtering
for col in features:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    df = df[(df[col] >= q1 - 1.5 * iqr) & (df[col] <= q3 + 1.5 * iqr)]

X = df[features]
y = df[target]
print('Processed dataset shape:', X.shape)

### 5. Train-Test Split & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape[0]} | Test: {X_test_sc.shape[0]}')

### 6. Model Training & 5-Fold Cross-Validation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree (Shallow)': DecisionTreeRegressor(
        max_depth=2, min_samples_split=80, min_samples_leaf=40, random_state=42
    ),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=8, min_samples_split=8, min_samples_leaf=4, max_features='sqrt', random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train_sc, y_train)
    cv_scores = cross_val_score(model, X_train_sc, y_train, cv=5, scoring='r2')
    print(f'{name:<24} -> 5-Fold CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

### 7. Benchmark & Evaluation Metrics

In [ ]:
results = []
predictions = {}

for name, model in models.items():
    y_pred = model.predict(X_test_sc)
    predictions[name] = y_pred
    
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100
    
    results.append({
        'Model': name,
        'Test R2': round(r2, 4),
        'RMSE (C)': round(rmse, 2),
        'MSE': round(mse, 2),
        'MAPE (%)': round(mape, 2)
    })

eval_df = pd.DataFrame(results)
display(eval_df)

### 8. Actual vs. Predicted & Residuals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'darkorange', 'forestgreen']

for idx, (name, preds) in enumerate(predictions.items()):
    ax = axes[idx]
    ax.scatter(y_test, preds, alpha=0.4, color=colors[idx], s=20)
    lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
    ax.plot(lims, lims, 'r--', lw=2, label='Ideal y = x')
    ax.set_title(f'{name}\nR2={r2_score(y_test, preds):.3f} | RMSE={np.sqrt(mean_squared_error(y_test, preds)):.1f} C')
    ax.set_xlabel('Actual Temp (C)')
    ax.set_ylabel('Predicted Temp (C)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
rf_residuals = y_test - predictions['Random Forest']

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.scatter(predictions['Random Forest'], rf_residuals, alpha=0.4, color='forestgreen', s=20)
plt.axhline(0, color='red', linestyle='--')
plt.title('Random Forest: Residuals vs Predicted')
plt.xlabel('Predicted Temp (C)')
plt.ylabel('Residual (C)')

plt.subplot(1, 2, 2)
sns.histplot(rf_residuals, bins=35, kde=True, color='forestgreen')
plt.axvline(0, color='red', linestyle='--')
plt.title('Residual Error Distribution')
plt.xlabel('Residual Error (C)')
plt.tight_layout()
plt.show()

### 9. Feature Importance (Random Forest)

In [ ]:
rf_model = models['Random Forest']
imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(9, 4))
sns.barplot(data=imp_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance Ranking')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

display(imp_df)

### 10. Model Serialization

In [ ]:
joblib.dump(models['Random Forest'], 'rf_model.pkl')
joblib.dump(models['Decision Tree (Shallow)'], 'dt_model.pkl')
joblib.dump(models['Linear Regression'], 'lr_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Models and scaler saved successfully.')

### 11. Inference Function

In [ ]:
def predict_cpu_temperature(cpu_util, mem_usage, clock_spd, amb_temp, volt, curr, model_name='Random Forest'):
    power_load = volt * curr
    workload = cpu_util * mem_usage
    
    input_df = pd.DataFrame([[cpu_util, mem_usage, clock_spd, amb_temp, volt, curr, power_load, workload]], columns=features)
    input_sc = scaler.transform(input_df)
    
    pred = models[model_name].predict(input_sc)[0]
    status = 'NORMAL (< 60C)' if pred < 60 else ('WARNING (60C - 80C)' if pred < 80 else 'CRITICAL (> 80C)')
    return round(pred, 2), status

# Sample prediction test
temp, status = predict_cpu_temperature(cpu_util=75, mem_usage=60, clock_spd=3.0, amb_temp=28, volt=1.2, curr=25)
print(f'Predicted CPU Temperature: {temp} C | Thermal Alert: {status}')